# 01 · Ingesta y adquisición de datos
**Proyecto:** Minería de datos · Contratación pública de la Alcaldía de Pereira

**Pregunta de negocio (Ficha):** ¿Qué procesos de contratación de la Alcaldía de Pereira registrados en SECOP II entre 2022 y 2025 presentan señales de concentración en proveedores o montos atípicos, para que la Oficina de Control Interno priorice cuáles revisar primero?

**Objetivo de este notebook:** leer las dos fuentes, cruzarlas verificando el número de filas y documentar el dataset resultante.

## Fuentes
| # | Fuente | Archivo | Periodo | Contenido |
|---|---|---|---|---|
| 1 | SECOP II · Contratos Electrónicos (datos.gov.co, `jbjy-vk9h`) | `Vista-_Contratos_Alcaldia_de_Pereira_2020-2023.xlsx` y `Vista-_Contratos_Alcaldia_de_Pereira_2024-2026.xlsx` | 2020-2026 | Un registro por contrato firmado: proveedor, valor, fechas, modalidad |
| 2 | SECOP II · Procesos de Contratación (datos.gov.co, `p6dx-8zbt`) | `procesos_pereira_p6dx-8zbt.csv` | 2017-2026 | Un registro por proceso: dependencia que contrata, precio base, número de oferentes, justificación de la modalidad |

**Por qué estas dos fuentes:** los contratos dicen *a quién* se le pagó y *cuánto*; los procesos dicen *cómo* se escogió al proveedor (cuántos se presentaron, qué dependencia contrató, con qué precio de referencia). Para detectar concentración y montos atípicos se necesitan las dos miradas.

**Nota metodológica:** la fuente 1 se exportó en dos archivos (2020-2023 y 2024-2026) para comparar las dos administraciones; primero se unen y luego se cruzan con la fuente 2 por el código del proceso.

In [1]:
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Lectura de las fuentes
### 1.1 Fuente 1 · Contratos
Se leen los dos archivos con `pd.read_excel`. La función `buscar` los localiza en la carpeta actual (Colab) o en `data/raw` (repositorio). Se verifica con `.shape` (filas, columnas) y `.head()`.

In [2]:
def buscar(nombre):
    for base in [".", "data/raw", "../data/raw"]:
        ruta = Path(base) / nombre
        if ruta.exists():
            return ruta
    raise FileNotFoundError(f"No se encontró {nombre}. Súbelo a Colab o colócalo en data/raw.")

ruta1 = buscar("Vista-_Contratos_Alcaldia_de_Pereira_2020-2023.xlsx")
ruta2 = buscar("Vista-_Contratos_Alcaldia_de_Pereira_2024-2026.xlsx")

df1 = pd.read_excel(ruta1)
df2 = pd.read_excel(ruta2)

print("Fuente 1 (2020-2023):", df1.shape)
print("Fuente 2 (2024-2026):", df2.shape)

Fuente 1 (2020-2023): (22522, 14)
Fuente 2 (2024-2026): (17753, 14)


In [3]:
display(df1.head())
display(df2.head())

,Proceso de Compra,ID Contrato,Referencia del Contrato,Estado Contrato,Descripcion del Proceso,Tipo de Contrato,Modalidad de Contratacion,Fecha de Inicio del Contrato,Fecha de Fin del Contrato,Documento Proveedor,Proveedor Adjudicado,Valor del Contrato,URLProceso,Objeto del Contrato
0,CO1.BDOS.2170640,CO1.PCCNTR.2759094,4095,Cerrado,PRESTACION DE SERVICIOS PROFESIONALES PARA DES...,Prestación de servicios,Contratación directa,2021-08-21,2021-12-28,1088262230,JULIO ANDRES TREJOS BUITRAGO,17874000.0,https://community.secop.gov.co/Public/Tenderin...,PRESTACION DE SERVICIOS PROFESIONALES PARA DES...
1,CO1.BDOS.2163731,CO1.PCCNTR.2750283,4052.2021,Cerrado,PRESTACION DE SERVICIOS PROFESIONALES RELACION...,Prestación de servicios,Contratación directa,2021-08-13,2021-12-28,1093226039,Luis Miguel García Rendón,18006400.0,https://community.secop.gov.co/Public/Tenderin...,PRESTACION DE SERVICIOS PROFESIONALES RELACION...
2,CO1.BDOS.2164424,CO1.PCCNTR.2751805,4086-2021,Cerrado,PRESTACION DE SERVICIOS DE APOYO PARA REALIZAR...,Prestación de servicios,Contratación directa,2021-08-11,2021-12-30,9958487,JUAN PABLO SEPULVEDA RESTREPO,5451000.0,https://community.secop.gov.co/Public/Tenderin...,PRESTACION DE SERVICIOS DE APOYO PARA REALIZAR...
3,CO1.BDOS.4103726,CO1.PCCNTR.4717199,2768-2023,Cerrado,PRESTACIÓN DE SERVICIOS PROFESIONALES PARA EJE...,Prestación de servicios,Contratación directa,2023-03-03,2023-12-22,9861149,Eduardo Arias Pineda,45394666.0,https://community.secop.gov.co/Public/Tenderin...,PRESTACIÓN DE SERVICIOS PROFESIONALES PARA EJE...
4,CO1.BDOS.2306534,CO1.PCCNTR.2997288,5584-2021,En ejecución,PUBLICACIÓN EN PRENSA DE AVISOS Y REQUERIMIENT...,Otro,Mínima cuantía,2021-11-12,2021-12-31,891411166,RR EDITORES RAMÍREZ Y RAMIREZ S.A.S.,9100000.0,https://community.secop.gov.co/Public/Tenderin...,PUBLICACIÓN EN PRENSA DE AVISOS Y REQUERIMIENT...


,Proceso de Compra,ID Contrato,Referencia del Contrato,Estado Contrato,Descripcion del Proceso,Tipo de Contrato,Modalidad de Contratacion,Fecha de Inicio del Contrato,Fecha de Fin del Contrato,Documento Proveedor,Proveedor Adjudicado,Valor del Contrato,URLProceso,Objeto del Contrato
0,CO1.BDOS.5925455,CO1.PCCNTR.6173597,2307-2024,En ejecución,PRESTAR SERVICIOS PROFESIONALES ESPECIALIZADOS...,Prestación de servicios,Contratación directa,2024-04-10,2024-10-09,10032397,Jesus Damian Hernandez Correa,28200000.0,https://community.secop.gov.co/Public/Tenderin...,PRESTAR SERVICIOS PROFESIONALES ESPECIALIZADOS...
1,CO1.BDOS.7877983,CO1.PCCNTR.7699502,3066-2025,Modificado,PRESTACION DE SERVICIOS PROFESIONALES PARA EL ...,Prestación de servicios,Contratación directa,2025-04-03,2025-12-19,1112930886,Viviana María Cortes Gómez,32553333.0,https://community.secop.gov.co/Public/Tenderin...,PRESTACION DE SERVICIOS PROFESIONALES PARA EL ...
2,CO1.BDOS.6047678,CO1.PCCNTR.6290237,2814 - 2024,En ejecución,Prestar servicios de apoyo a la gestión para r...,Prestación de servicios,Contratación directa,2024-05-14,2024-12-15,42098660,GLORIA NANCY GUTIERREZ GOMEZ,13020000.0,https://community.secop.gov.co/Public/Tenderin...,Prestar servicios de apoyo a la gestión para r...
3,CO1.BDOS.6068007,CO1.PCCNTR.6290241,2836 - 2024,En ejecución,Prestar servicios de apoyo a la gestión para r...,Prestación de servicios,Contratación directa,2024-05-14,2024-12-31,10144057,JOSE ORLANDO HOLGUIN ACEVEDO,13980000.0,https://community.secop.gov.co/Public/Tenderin...,Prestar servicios de apoyo a la gestión para r...
4,CO1.BDOS.7886745,CO1.PCCNTR.7707232,3042-2025,Modificado,PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN ...,Prestación de servicios,Contratación directa,2025-04-02,2025-12-19,1092913113,Fredy Alejandro Martinez,24510000.0,https://community.secop.gov.co/Public/Tenderin...,PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN ...


In [4]:
print("Columnas fuente 1:\n", list(df1.columns), "\n")
print("Columnas fuente 2:\n", list(df2.columns), "\n")

comunes = set(df1.columns) & set(df2.columns)
print("Columnas en común:", len(comunes))
print("Solo en fuente 1:", set(df1.columns) - set(df2.columns))
print("Solo en fuente 2:", set(df2.columns) - set(df1.columns))

df1.info()

Columnas fuente 1:
 ['Proceso de Compra', 'ID Contrato', 'Referencia del Contrato', 'Estado Contrato', 'Descripcion del Proceso', 'Tipo de Contrato', 'Modalidad de Contratacion', 'Fecha de Inicio del Contrato', 'Fecha de Fin del Contrato', 'Documento Proveedor', 'Proveedor Adjudicado', 'Valor del Contrato', 'URLProceso', 'Objeto del Contrato'] 

Columnas fuente 2:
 ['Proceso de Compra', 'ID Contrato', 'Referencia del Contrato', 'Estado Contrato', 'Descripcion del Proceso', 'Tipo de Contrato', 'Modalidad de Contratacion', 'Fecha de Inicio del Contrato', 'Fecha de Fin del Contrato', 'Documento Proveedor', 'Proveedor Adjudicado', 'Valor del Contrato', 'URLProceso', 'Objeto del Contrato'] 

Columnas en común: 14
Solo en fuente 1: set()
Solo en fuente 2: set()
<class 'pandas.DataFrame'>
RangeIndex: 22522 entries, 0 to 22521
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0  

### 1.2 Fuente 2 · Procesos
El dataset de Procesos tiene millones de filas de todo el país, así que no se descarga completo: se consulta la API pública de datos.gov.co filtrando por el NIT del Municipio de Pereira (`891480030`). El resultado se guardó en `data/raw/procesos_pereira_p6dx-8zbt.csv` con el script `src/descargar_procesos.py`. Si el archivo no está (por ejemplo, en Colab), la celda lo descarga con la misma consulta.

In [5]:
ARCHIVO_PROCESOS = "procesos_pereira_p6dx-8zbt.csv"

# Mismas columnas, filtro y orden que src/descargar_procesos.py
COLUMNAS_PROCESOS = [
    "id_del_portafolio", "id_del_proceso", "referencia_del_proceso", "entidad", "nit_entidad",
    "nombre_de_la_unidad_de", "nombre_del_procedimiento", "modalidad_de_contratacion",
    "justificaci_n_modalidad_de", "tipo_de_contrato", "fase", "estado_del_procedimiento",
    "fecha_de_publicacion_del", "precio_base", "duracion", "unidad_de_duracion",
    "proveedores_invitados", "proveedores_que_manifestaron", "respuestas_al_procedimiento",
    "proveedores_unicos_con", "adjudicado", "valor_total_adjudicacion",
]

def descargar_procesos():
    """Misma consulta de src/descargar_procesos.py: procesos del Municipio de Pereira en SECOP II."""
    import json, urllib.parse, urllib.request
    url = "https://www.datos.gov.co/resource/p6dx-8zbt.json"
    filas, offset = [], 0
    while True:
        params = {"$select": ",".join(COLUMNAS_PROCESOS), "$where": "nit_entidad='891480030'",
                  "$order": "id_del_proceso", "$limit": 50000, "$offset": offset}
        with urllib.request.urlopen(f"{url}?{urllib.parse.urlencode(params)}", timeout=300) as r:
            pagina = json.load(r)
        filas.extend(pagina)
        if len(pagina) < 50000:
            return pd.DataFrame(filas, columns=COLUMNAS_PROCESOS)
        offset += 50000

try:
    df_proc = pd.read_csv(buscar(ARCHIVO_PROCESOS), low_memory=False)
    print("Procesos leídos desde", buscar(ARCHIVO_PROCESOS))
except FileNotFoundError:
    df_proc = descargar_procesos()
    df_proc.to_csv(ARCHIVO_PROCESOS, index=False)
    print("Procesos descargados de la API de datos.gov.co")

print("Fuente 2 (procesos):", df_proc.shape)

Procesos leídos desde ..\data\raw\procesos_pereira_p6dx-8zbt.csv
Fuente 2 (procesos): (43863, 22)


In [6]:
display(df_proc.head())
print("Entidades en el archivo:", df_proc["entidad"].unique())
df_proc.info()

,id_del_portafolio,id_del_proceso,referencia_del_proceso,entidad,nit_entidad,nombre_de_la_unidad_de,nombre_del_procedimiento,modalidad_de_contratacion,justificaci_n_modalidad_de,tipo_de_contrato,fase,estado_del_procedimiento,fecha_de_publicacion_del,precio_base,duracion,unidad_de_duracion,proveedores_invitados,proveedores_que_manifestaron,respuestas_al_procedimiento,proveedores_unicos_con,adjudicado,valor_total_adjudicacion
0,CO1.BDOS.9831388,CO1.REQ.10000025,SMS-CD-3452-2026,MUNICIPIO DE PEREIRA- OFICIAL,891480030,SECRETARIA DE SALUD,PRESTACION DE SERVICIOS DE APOYO A LA GESTION,Contratación directa,Servicios profesionales y apoyo a la gestión,Prestación de servicios,Presentación de oferta,Seleccionado,2026-01-28T00:00:00.000,13000000,5,Mes(es),0,0,0,0,No,0
1,CO1.BDOS.9830900,CO1.REQ.10000039,SMH-CI-0767-2026,MUNICIPIO DE PEREIRA- OFICIAL,891480030,SECRETARIA DE HACIENDA Y FINANZAS PÚBLICAS,PRESTACION DE SERVICIOS EN LA RECEPCION DE LAS...,Contratación directa,Contratos o convenios Interadministrativos (va...,Otro,Presentación de oferta,Seleccionado,2026-01-28T00:00:00.000,50000000,330,día(s),0,0,0,0,No,0
2,CO1.BDOS.9832131,CO1.REQ.10000293,SMDEC-CD-2944-2026,MUNICIPIO DE PEREIRA- OFICIAL,891480030,SECRETARIA DE DESARROLLO ECONÓMICO Y COMPETITI...,PRESTACIÓN DE SERVICIOS PROFESIONALES ESPECIAL...,Contratación directa,Servicios profesionales y apoyo a la gestión,Prestación de servicios,Presentación de oferta,Seleccionado,2026-01-28T00:00:00.000,26250000,150,día(s),0,0,0,0,No,0
3,CO1.BDOS.9832020,CO1.REQ.10000407,SME-CD-3224-2026,MUNICIPIO DE PEREIRA- OFICIAL,891480030,SECRETARIA DE EDUCACION,COBERTURA EDUCATIVA,Contratación directa,Servicios profesionales y apoyo a la gestión,Prestación de servicios,Presentación de oferta,Seleccionado,2026-01-28T00:00:00.000,26250000,5,Mes(es),0,0,0,0,No,0
4,CO1.BDOS.9831631,CO1.REQ.10000509,SMC-CD-2890-2026,MUNICIPIO DE PEREIRA- OFICIAL,891480030,SECRETARIA DE CULTURA,PRESTACION DE SERVICIOS PROFESIONALES PARA APO...,Contratación directa,Servicios profesionales y apoyo a la gestión,Prestación de servicios,Presentación de oferta,Seleccionado,2026-01-28T00:00:00.000,20000000,5,Mes(es),0,0,0,0,No,0


Entidades en el archivo: <StringArray>
['MUNICIPIO DE PEREIRA- OFICIAL']
Length: 1, dtype: str
<class 'pandas.DataFrame'>
RangeIndex: 43863 entries, 0 to 43862
Data columns (total 22 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   id_del_portafolio             43863 non-null  str  
 1   id_del_proceso                43863 non-null  str  
 2   referencia_del_proceso        43863 non-null  str  
 3   entidad                       43863 non-null  str  
 4   nit_entidad                   43863 non-null  int64
 5   nombre_de_la_unidad_de        43863 non-null  str  
 6   nombre_del_procedimiento      43863 non-null  str  
 7   modalidad_de_contratacion     43863 non-null  str  
 8   justificaci_n_modalidad_de    43863 non-null  str  
 9   tipo_de_contrato              43863 non-null  str  
 10  fase                          43372 non-null  str  
 11  estado_del_procedimiento      43863 non-null  str  
 12  fech

## 2. Normalización de columnas de la fuente 1
Se pasan los nombres a minúsculas, sin tildes ni espacios, y se agrega `fuente_periodo` para saber de qué archivo viene cada fila.

Las columnas de la fuente 2 ya vienen en ese formato desde la API; se renombran en la sección 5.

In [7]:
def normalizar(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(" ", "_")
                  .str.replace("[áàä]", "a", regex=True)
                  .str.replace("[éèë]", "e", regex=True)
                  .str.replace("[íìï]", "i", regex=True)
                  .str.replace("[óòö]", "o", regex=True)
                  .str.replace("[úùü]", "u", regex=True))
    return df

df1 = normalizar(df1); df1["fuente_periodo"] = "2020-2023"
df2 = normalizar(df2); df2["fuente_periodo"] = "2024-2026"
print(list(df1.columns))

['proceso_de_compra', 'id_contrato', 'referencia_del_contrato', 'estado_contrato', 'descripcion_del_proceso', 'tipo_de_contrato', 'modalidad_de_contratacion', 'fecha_de_inicio_del_contrato', 'fecha_de_fin_del_contrato', 'documento_proveedor', 'proveedor_adjudicado', 'valor_del_contrato', 'urlproceso', 'objeto_del_contrato', 'fuente_periodo']


## 3. Unión de los dos periodos de contratos
Antes del cruce, los dos archivos de la fuente 1 se unen en uno solo.

**Columna de control:** `id_contrato`, identificador único de cada contrato. Se revisa que no tenga nulos ni duplicados y que ningún contrato aparezca en los dos periodos; luego se apilan con `pd.concat`. Se verifica el número de filas **antes y después**.

In [8]:
LLAVE = "id_contrato"

print("Nulos llave  -> df1:", df1[LLAVE].isna().sum(), "| df2:", df2[LLAVE].isna().sum())
print("Duplicados   -> df1:", df1[LLAVE].duplicated().sum(), "| df2:", df2[LLAVE].duplicated().sum())
print("Contratos presentes en AMBOS periodos:", len(set(df1[LLAVE]) & set(df2[LLAVE])))

Nulos llave  -> df1: 0 | df2: 0
Duplicados   -> df1: 0 | df2: 0
Contratos presentes en AMBOS periodos: 0


In [9]:
print("Filas ANTES -> df1:", len(df1), "| df2:", len(df2), "| suma:", len(df1) + len(df2))

cruce = pd.merge(df1[[LLAVE, "fuente_periodo"]],
                 df2[[LLAVE, "fuente_periodo"]],
                 on=LLAVE, how="outer",
                 suffixes=("_a", "_b"), indicator=True)

print("Filas DESPUÉS del merge:", len(cruce))
print(cruce["_merge"].value_counts())
# left_only = solo en 2020-2023 | right_only = solo en 2024-2026 | both = en los dos

Filas ANTES -> df1: 22522 | df2: 17753 | suma: 40275


Filas DESPUÉS del merge: 40275
_merge
left_only     22522
right_only    17753
both              0
Name: count, dtype: int64


In [10]:
df_final = pd.concat([df1, df2], ignore_index=True)
print("Filas tras concat:", len(df_final))

df_final = df_final.drop_duplicates(subset=LLAVE, keep="last")
print("Filas finales (sin ID repetido):", len(df_final))
assert df_final[LLAVE].is_unique

Filas tras concat: 40275
Filas finales (sin ID repetido): 40275


**Interpretación:** los dos archivos tienen el mismo esquema (14 columnas idénticas) y son partes de una misma exportación dividida por periodo. El `merge` outer por `id_contrato` dio 22.522 contratos `left_only`, 17.753 `right_only` y 0 `both`: no hay contratos repetidos entre periodos. Por eso la unión correcta es apilarlos (`pd.concat`): 22.522 + 17.753 = 40.275 filas, y `drop_duplicates` sobre la llave no eliminó ninguna.

Esto **no** es el cruce entre fuentes: es la preparación de la fuente 1. El cruce con la fuente 2 viene en la sección 5.

## 4. Tipos de dato de la fuente 1
Se convierten fechas y valores a su tipo correcto. `documento_proveedor` se deja como texto porque es un identificador, no una cantidad.

In [11]:
df_final["fecha_de_inicio_del_contrato"] = pd.to_datetime(df_final["fecha_de_inicio_del_contrato"], errors="coerce")
df_final["fecha_de_fin_del_contrato"]    = pd.to_datetime(df_final["fecha_de_fin_del_contrato"], errors="coerce")
df_final["valor_del_contrato"]  = pd.to_numeric(df_final["valor_del_contrato"], errors="coerce")
df_final["documento_proveedor"] = df_final["documento_proveedor"].astype(str)

print(df_final.shape)
df_final.dtypes

(40275, 15)


proceso_de_compra                          str
id_contrato                                str
referencia_del_contrato                    str
estado_contrato                            str
descripcion_del_proceso                    str
tipo_de_contrato                           str
modalidad_de_contratacion                  str
fecha_de_inicio_del_contrato    datetime64[us]
fecha_de_fin_del_contrato       datetime64[us]
documento_proveedor                        str
proveedor_adjudicado                       str
valor_del_contrato                     float64
urlproceso                                 str
objeto_del_contrato                        str
fuente_periodo                             str
dtype: object

In [12]:
# Para describir bien las columnas categóricas en el diccionario
print(df_final["estado_contrato"].unique())
print(df_final["tipo_de_contrato"].unique())
print(df_final["modalidad_de_contratacion"].unique())

<StringArray>
['Cerrado', 'En ejecución', 'Modificado', 'terminado', 'cedido', 'Cancelado', 'Aprobado', 'Borrador', 'Suspendido', 'enviado Proveedor']
Length: 10, dtype: str
<StringArray>
[       'Prestación de servicios',                           'Otro',            'Decreto 092 de 2017',                           'Obra',     'Arrendamiento de inmuebles',
                    'Suministros',                    'Consultoría',       'Arrendamiento de muebles',                    'Compraventa',                       'Comodato',
                  'Interventoría',          'Servicios financieros',                        'Seguros', 'Operaciones de Crédito Público',                      'Concesión']
Length: 15, dtype: str
<StringArray>
[                                       'Contratación directa',                                              'Mínima cuantía',                 'Contratación régimen especial (con ofertas)',
                               'Contratación régimen especial',         

## 5. Cruce de las fuentes: Contratos ↔ Procesos
**Columna en común (llave):** el código del proceso de compra, con formato `CO1.BDOS.<número>`. En Contratos se llama `proceso_de_compra` y en Procesos `id_del_portafolio`. Cada contrato nace de un proceso, así que la relación esperada es **muchos contratos → un proceso**.

### 5.1 Preparación de la fuente 2
Se dejan las columnas útiles para la pregunta, se renombran con nombres claros y se convierten los tipos. `modalidad_de_contratacion` y `tipo_de_contrato` también existen en Contratos: se conservan con sufijo `_proceso` solo para validar la llave (5.4).

In [13]:
assert df_proc["nit_entidad"].astype(str).eq("891480030").all(), "Hay procesos de otra entidad"

renombrar = {
    "id_del_portafolio": "id_del_portafolio",
    "id_del_proceso": "id_del_proceso",
    "referencia_del_proceso": "referencia_del_proceso",
    "nombre_de_la_unidad_de": "dependencia",
    "justificaci_n_modalidad_de": "justificacion_modalidad",
    "fase": "fase_proceso",
    "estado_del_procedimiento": "estado_proceso",
    "fecha_de_publicacion_del": "fecha_publicacion_proceso",
    "precio_base": "precio_base",
    "duracion": "duracion_proceso",
    "unidad_de_duracion": "unidad_duracion_proceso",
    "proveedores_invitados": "proveedores_invitados",
    "proveedores_que_manifestaron": "proveedores_que_manifestaron",
    "respuestas_al_procedimiento": "respuestas_al_procedimiento",
    "proveedores_unicos_con": "proveedores_unicos_con_respuesta",
    "modalidad_de_contratacion": "modalidad_de_contratacion_proceso",
    "tipo_de_contrato": "tipo_de_contrato_proceso",
}
proc = df_proc[list(renombrar)].rename(columns=renombrar)

proc["fecha_publicacion_proceso"] = pd.to_datetime(proc["fecha_publicacion_proceso"], errors="coerce")
proc["precio_base"] = pd.to_numeric(proc["precio_base"], errors="coerce")
for col in ["duracion_proceso", "proveedores_invitados", "proveedores_que_manifestaron",
            "respuestas_al_procedimiento", "proveedores_unicos_con_respuesta"]:
    proc[col] = pd.to_numeric(proc[col], errors="coerce").astype("Int64")

print("Procesos:", proc.shape)

Procesos: (43863, 17)


### 5.2 Calidad de la llave en Procesos
En Contratos la llave puede repetirse (un proceso puede dar varios contratos). En Procesos debería ser única; si no lo es, el `merge` multiplicaría filas.

In [14]:
LLAVE_C, LLAVE_P = "proceso_de_compra", "id_del_portafolio"

print("Nulos llave        -> contratos:", df_final[LLAVE_C].isna().sum(), "| procesos:", proc[LLAVE_P].isna().sum())
print("Filas en procesos:", len(proc), "| códigos de proceso distintos:", proc[LLAVE_P].nunique())
print("Filas 100% idénticas en procesos:", proc.duplicated().sum())

repetidos = proc[proc[LLAVE_P].duplicated(keep=False)]
print("Códigos que aparecen más de una vez:", repetidos[LLAVE_P].nunique())
print("  ...de ellos, con varias fases (distinto id_del_proceso):",
      (repetidos.groupby(LLAVE_P)["id_del_proceso"].nunique() > 1).sum())

Nulos llave        -> contratos: 0 | procesos: 0


Filas en procesos: 43863 | códigos de proceso distintos: 40746


Filas 100% idénticas en procesos: 2450


Códigos que aparecen más de una vez: 1556
  ...de ellos, con varias fases (distinto id_del_proceso): 522


In [15]:
# Ejemplo: un mismo proceso publicado en tres fases
ejemplo = repetidos.groupby(LLAVE_P)["id_del_proceso"].nunique().idxmax()
proc.loc[proc[LLAVE_P] == ejemplo,
         [LLAVE_P, "id_del_proceso", "fase_proceso", "estado_proceso", "fecha_publicacion_proceso", "precio_base"]]

,id_del_portafolio,id_del_proceso,fase_proceso,estado_proceso,fecha_publicacion_proceso,precio_base
26555,CO1.BDOS.4671126,CO1.REQ.4786456,Presentación de observaciones,Evaluación,2023-07-10,600000000
26700,CO1.BDOS.4671126,CO1.REQ.4847730,Manifestación de interés (Menor Cuantía),Evaluación,2023-07-19,600000000
26745,CO1.BDOS.4671126,CO1.REQ.4864928,NaN,Cancelado,2023-07-19,600000000
26815,CO1.BDOS.4671126,CO1.REQ.4879989,Presentación de oferta,Seleccionado,2023-07-27,600000000
26816,CO1.BDOS.4671126,CO1.REQ.4879990,Presentación de oferta,Cancelado,2023-07-27,600000000


### 5.3 Qué pasa si se cruza sin depurar
Se hace el `merge` con los procesos tal como vienen, para medir el efecto de la llave repetida.

In [16]:
ingenuo = pd.merge(df_final, proc, left_on=LLAVE_C, right_on=LLAVE_P, how="left")
print("Contratos ANTES:", len(df_final), "| filas DESPUÉS del merge sin depurar:", len(ingenuo),
      f"(+{len(ingenuo) - len(df_final)} filas duplicadas)")

Contratos ANTES: 40275 | filas DESPUÉS del merge sin depurar: 45384 (+5109 filas duplicadas)


El cruce ingenuo **inventa contratos**: cada contrato cuyo proceso tiene varias filas aparece repetido, y cualquier suma de `valor_del_contrato` quedaría inflada.

**Depuración:** se eliminan las filas idénticas y, cuando un proceso tiene varias fases, se conserva **la última publicada** (fecha más reciente y, si empatan, el `id_del_proceso` mayor), que es la que describe el proceso tal como terminó. En 31 procesos una de las fases no tiene fecha (suelen ser borradores o publicaciones canceladas); esas se ordenan primero para que nunca desplacen a una fase fechada.

In [17]:
filas_antes = len(proc)
proc = proc.drop_duplicates()
print("Sin filas idénticas:", len(proc), f"(-{filas_antes - len(proc)})")

proc["_n_req"] = proc["id_del_proceso"].str.extract(r"(\d+)\D*$")[0].astype("int64")  # un id viene como CO1.REQ.452959_
# na_position="first": una publicación sin fecha (borrador o cancelada) nunca gana a una fechada
proc = (proc.sort_values([LLAVE_P, "fecha_publicacion_proceso", "_n_req"], na_position="first")
            .drop_duplicates(subset=LLAVE_P, keep="last")
            .drop(columns="_n_req"))
print("Un registro por proceso:", len(proc))
assert proc[LLAVE_P].is_unique

Sin filas idénticas: 41413 (-2450)


Un registro por proceso: 40746


### 5.4 Cruce y verificación de filas
`merge` **left**: se conservan todos los contratos y se les agrega la información de su proceso. `validate="many_to_one"` hace que pandas falle si la llave de Procesos no fuera única, e `indicator=True` marca qué contratos encontraron su proceso.

In [18]:
print("Filas ANTES  -> contratos:", len(df_final), "| procesos:", len(proc))

cruzado = pd.merge(df_final, proc, left_on=LLAVE_C, right_on=LLAVE_P,
                   how="left", validate="many_to_one", indicator=True)

print("Filas DESPUÉS del merge:", len(cruzado))
assert len(cruzado) == len(df_final), "El cruce cambió el número de contratos"
print(cruzado["_merge"].value_counts())
print(f"\nContratos con proceso encontrado: {(cruzado['_merge'] == 'both').mean():.1%}")

Filas ANTES  -> contratos: 40275 | procesos: 40746


Filas DESPUÉS del merge: 40275
_merge
both          37523
left_only      2752
right_only        0
Name: count, dtype: int64

Contratos con proceso encontrado: 93.2%


In [19]:
# Validación de la llave: si el cruce es correcto, lo que ambas fuentes dicen del mismo contrato debe coincidir
con_proc = cruzado[cruzado["_merge"] == "both"]
print("Modalidad igual en ambas fuentes:",
      f"{(con_proc['modalidad_de_contratacion'] == con_proc['modalidad_de_contratacion_proceso']).mean():.1%}")
print("Tipo de contrato igual en ambas fuentes:",
      f"{(con_proc['tipo_de_contrato'] == con_proc['tipo_de_contrato_proceso']).mean():.1%}")

# Ya validadas, las columnas repetidas y la llave duplicada sobran
cruzado["tiene_proceso"] = cruzado["_merge"] == "both"
df_cruzado = cruzado.drop(columns=["_merge", LLAVE_P, "modalidad_de_contratacion_proceso", "tipo_de_contrato_proceso"])
print("\nDataset cruzado:", df_cruzado.shape)

Modalidad igual en ambas fuentes: 100.0%
Tipo de contrato igual en ambas fuentes: 100.0%

Dataset cruzado: (40275, 30)


In [20]:
# ¿Quiénes son los contratos sin proceso?
sin_proc = cruzado[cruzado["_merge"] == "left_only"]
print("Contratos sin proceso:", len(sin_proc))
print("\nPor modalidad:"); print(sin_proc["modalidad_de_contratacion"].value_counts())
print("\nPor año de inicio:"); print(sin_proc["fecha_de_inicio_del_contrato"].dt.year.value_counts().sort_index())

Contratos sin proceso: 2752

Por modalidad:
modalidad_de_contratacion
Contratación directa                  2750
Contratación Directa (con ofertas)       2
Name: count, dtype: int64

Por año de inicio:
fecha_de_inicio_del_contrato
2020     152
2021      10
2022      30
2023      13
2024    1314
2025     515
2026     718
Name: count, dtype: int64


**Interpretación del cruce:**
- El cruce ingenuo producía más de 5.000 filas de más porque la fuente 2 publica el mismo proceso varias veces (filas idénticas y una fila por cada fase). Tras depurar a un registro por proceso, el `merge` conserva exactamente los 40.275 contratos.
- La llave es válida: en todos los contratos cruzados, la modalidad y el tipo de contrato coinciden al 100 % entre las dos fuentes, que se construyen por separado.
- Alrededor del 93 % de los contratos encontró su proceso. Los que no lo encontraron son casi todos de **contratación directa** y se concentran desde 2024. Se buscaron en el dataset completo de Procesos (sin filtrar por NIT) y no aparecen: es un vacío de publicación de SECOP II, no un error del cruce. Se conservan con `tiene_proceso = False` para no sesgar el análisis de proveedores.
- El cruce agrega lo que la fuente 1 no tenía para la pregunta de la Ficha: la **dependencia** que contrata, el **precio base** y el **número de oferentes**, necesarios para detectar concentración y montos atípicos.

## 6. Diccionario de datos
Descripción de todas las columnas del dataset cruzado: tipo de dato, significado y fuente.

In [21]:
FUENTE = "SECOP II · Contratos Electrónicos (datos.gov.co, jbjy-vk9h)"
FUENTE_P = "SECOP II · Procesos de Contratación (datos.gov.co, p6dx-8zbt)"

significados = {
    "proceso_de_compra": ("Código del proceso de compra del que se origina el contrato (llave del cruce con Procesos)", FUENTE),
    "id_contrato": ("Identificador único del contrato", FUENTE),
    "referencia_del_contrato": ("Número o referencia interna asignada al contrato", FUENTE),
    "estado_contrato": ("Estado del contrato en la plataforma (Cerrado, En ejecución, Modificado, terminado, cedido, Cancelado, Aprobado, Borrador, Suspendido, enviado Proveedor)", FUENTE),
    "descripcion_del_proceso": ("Descripción general del proceso de compra", FUENTE),
    "tipo_de_contrato": ("Categoría del contrato (Prestación de servicios, Obra, Suministros, Consultoría, Arrendamiento, Interventoría, Seguros, Concesión, entre otras)", FUENTE),
    "modalidad_de_contratacion": ("Modalidad de selección (Contratación directa, Mínima cuantía, Licitación pública, Selección abreviada, Concurso de méritos, Régimen especial)", FUENTE),
    "fecha_de_inicio_del_contrato": ("Fecha de inicio de la ejecución del contrato", FUENTE),
    "fecha_de_fin_del_contrato": ("Fecha de terminación prevista del contrato", FUENTE),
    "documento_proveedor": ("NIT o cédula del proveedor adjudicado", FUENTE),
    "proveedor_adjudicado": ("Nombre del proveedor o contratista", FUENTE),
    "valor_del_contrato": ("Valor del contrato en pesos colombianos (COP)", FUENTE),
    "urlproceso": ("Enlace al proceso en la plataforma de contratación", FUENTE),
    "objeto_del_contrato": ("Descripción de lo que se contrata", FUENTE),
    "fuente_periodo": ("Archivo/periodo de origen del registro: 2020-2023 o 2024-2026", "Creada en este notebook"),
    "id_del_proceso": ("Identificador de la publicación del proceso (CO1.REQ); se conserva la última fase", FUENTE_P),
    "referencia_del_proceso": ("Referencia interna del proceso asignada por la Alcaldía", FUENTE_P),
    "dependencia": ("Secretaría o unidad de la Alcaldía que adelanta el proceso", FUENTE_P),
    "justificacion_modalidad": ("Causal legal que justifica la modalidad (p. ej. servicios profesionales y apoyo a la gestión)", FUENTE_P),
    "fase_proceso": ("Fase en la que quedó el proceso en su última publicación", FUENTE_P),
    "estado_proceso": ("Estado del procedimiento (Seleccionado, Evaluación, Cancelado, entre otros)", FUENTE_P),
    "fecha_publicacion_proceso": ("Fecha de publicación del proceso en SECOP II", FUENTE_P),
    "precio_base": ("Presupuesto oficial estimado del proceso en COP, antes de adjudicar", FUENTE_P),
    "duracion_proceso": ("Duración estimada del contrato según el proceso (ver unidad_duracion_proceso)", FUENTE_P),
    "unidad_duracion_proceso": ("Unidad de la duración: día(s), mes(es) o año(s)", FUENTE_P),
    "proveedores_invitados": ("Número de proveedores invitados a participar", FUENTE_P),
    "proveedores_que_manifestaron": ("Número de proveedores que manifestaron interés", FUENTE_P),
    "respuestas_al_procedimiento": ("Número de respuestas u ofertas recibidas", FUENTE_P),
    "proveedores_unicos_con_respuesta": ("Número de proveedores distintos que presentaron oferta (medida de competencia)", FUENTE_P),
    "tiene_proceso": ("True si el contrato encontró su proceso en la fuente 2", "Creada en este notebook (resultado del cruce)"),
}

faltan = set(df_cruzado.columns) - set(significados)
assert not faltan, f"Columnas sin describir: {faltan}"

diccionario = pd.DataFrame({
    "columna": df_cruzado.columns,
    "tipo_dato": df_cruzado.dtypes.astype(str).values,
    "significado": [significados[c][0] for c in df_cruzado.columns],
    "fuente": [significados[c][1] for c in df_cruzado.columns],
})
pd.set_option("display.max_colwidth", 120)
diccionario

,columna,tipo_dato,significado,fuente
0,proceso_de_compra,str,Código del proceso de compra del que se origina el contrato (llave del cruce con Procesos),"SECOP II · Contratos Electrónicos (datos.gov.co, jbjy-vk9h)"
1,id_contrato,str,Identificador único del contrato,"SECOP II · Contratos Electrónicos (datos.gov.co, jbjy-vk9h)"
2,referencia_del_contrato,str,Número o referencia interna asignada al contrato,"SECOP II · Contratos Electrónicos (datos.gov.co, jbjy-vk9h)"
3,estado_contrato,str,"Estado del contrato en la plataforma (Cerrado, En ejecución, Modificado, terminado, cedido, Cancelado, Aprobado, Bor...","SECOP II · Contratos Electrónicos (datos.gov.co, jbjy-vk9h)"
4,descripcion_del_proceso,str,Descripción general del proceso de compra,"SECOP II · Contratos Electrónicos (datos.gov.co, jbjy-vk9h)"
5,tipo_de_contrato,str,"Categoría del contrato (Prestación de servicios, Obra, Suministros, Consultoría, Arrendamiento, Interventoría, Segur...","SECOP II · Contratos Electrónicos (datos.gov.co, jbjy-vk9h)"
6,modalidad_de_contratacion,str,"Modalidad de selección (Contratación directa, Mínima cuantía, Licitación pública, Selección abreviada, Concurso de m...","SECOP II · Contratos Electrónicos (datos.gov.co, jbjy-vk9h)"
7,fecha_de_inicio_del_contrato,datetime64[us],Fecha de inicio de la ejecución del contrato,"SECOP II · Contratos Electrónicos (datos.gov.co, jbjy-vk9h)"
8,fecha_de_fin_del_contrato,datetime64[us],Fecha de terminación prevista del contrato,"SECOP II · Contratos Electrónicos (datos.gov.co, jbjy-vk9h)"
9,documento_proveedor,str,NIT o cédula del proveedor adjudicado,"SECOP II · Contratos Electrónicos (datos.gov.co, jbjy-vk9h)"


## 7. Verificaciones de calidad
Se comprueba que el rango de fechas de cada archivo corresponda a su periodo, se cuentan los nulos y se revisan fechas inconsistentes (inicio posterior a hoy o fin anterior al inicio). Las columnas que vienen de Procesos quedan vacías en los contratos sin proceso; eso es esperado.

In [22]:
print(df_cruzado.groupby("fuente_periodo")["fecha_de_inicio_del_contrato"].agg(["min", "max", "count"]))
print("\nNulos por columna:")
print(df_cruzado.isna().sum()[lambda x: x > 0])
print("\nContratos sin proceso:", (~df_cruzado["tiene_proceso"]).sum())

                      min        max  count
fuente_periodo                             
2020-2023      2020-02-14 2023-12-30  22522
2024-2026      2024-01-16 2026-12-26  17753

Nulos por columna:
id_del_proceso                      2752
referencia_del_proceso              2752
dependencia                         2752
justificacion_modalidad             2752
fase_proceso                        2752
estado_proceso                      2752
fecha_publicacion_proceso           2854
precio_base                         2752
duracion_proceso                    2752
unidad_duracion_proceso             2752
proveedores_invitados               2752
proveedores_que_manifestaron        2752
respuestas_al_procedimiento         2752
proveedores_unicos_con_respuesta    2752
dtype: int64

Contratos sin proceso: 2752


In [23]:
hoy = pd.Timestamp.today()
futuros = df_cruzado["fecha_de_inicio_del_contrato"] > hoy
print("Contratos con inicio posterior a hoy:", futuros.sum())
print(df_cruzado.loc[futuros, "estado_contrato"].value_counts())
print("\nContratos con fin anterior al inicio:",
      (df_cruzado["fecha_de_fin_del_contrato"] < df_cruzado["fecha_de_inicio_del_contrato"]).sum())

Contratos con inicio posterior a hoy: 2
estado_contrato
En ejecución    2
Name: count, dtype: int64

Contratos con fin anterior al inicio: 3


## 7. Subconjunto de análisis 2022-2025
La pregunta de la Ficha cubre 2022-2025, así que se crea un subconjunto por fecha de inicio. El dataset completo (2020-2026) se conserva para contexto.

In [24]:
inicio = df_cruzado["fecha_de_inicio_del_contrato"]
df_analisis = df_cruzado[(inicio >= "2022-01-01") & (inicio <= "2025-12-31")].copy()

print("Dataset completo:", df_cruzado.shape)
print("Subconjunto 2022-2025:", df_analisis.shape)
print(df_analisis["fuente_periodo"].value_counts())
print(f"Con proceso encontrado: {df_analisis['tiene_proceso'].mean():.1%}")

Dataset completo: (40275, 30)
Subconjunto 2022-2025: (24779, 30)
fuente_periodo
2020-2023    13162
2024-2026    11617
Name: count, dtype: int64
Con proceso encontrado: 92.4%


## 9. Exportación
Se guardan el dataset cruzado, el subconjunto de análisis y el diccionario de datos en `data/processed` para las siguientes etapas (EDA y detección de anomalías). En Colab se guardan en la carpeta actual.

In [25]:
destino = next((Path(p) for p in ["../data/processed", "data/processed"] if Path(p).is_dir()), Path("."))

df_cruzado.to_csv(destino / "contratos_procesos_pereira_2020_2026.csv", index=False)
df_analisis.to_csv(destino / "contratos_procesos_pereira_2022_2025.csv", index=False)
diccionario.to_csv(destino / "diccionario_datos.csv", index=False)
print("Archivos guardados en", destino.resolve())

Archivos guardados en C:\Users\alber\Documents\mineria-datos\data\processed


## 10. Conclusiones de la etapa de adquisición
- **Fuente 1 (Contratos, `jbjy-vk9h`):** 2 archivos con 22.522 (2020-2023) y 17.753 (2024-2026) filas y 14 columnas idénticas. Se unieron con `pd.concat` en 40.275 contratos, sin contratos repetidos entre periodos.
- **Fuente 2 (Procesos, `p6dx-8zbt`):** 43.863 publicaciones del Municipio de Pereira descargadas de la API de datos.gov.co.
- **Cruce:** `merge` left por `proceso_de_compra` = `id_del_portafolio`. Filas antes: 40.275 contratos; filas después: 40.275. Sin depurar la fuente 2 el cruce inflaba el resultado en más de 5.000 filas; la depuración y `validate="many_to_one"` lo impiden.
- **Validez de la llave:** modalidad y tipo de contrato coinciden al 100 % entre las fuentes en los contratos cruzados. Cerca del 93 % de los contratos encontró su proceso.
- **Dataset final:** 40.275 filas y 30 columnas, documentadas en el diccionario de datos. El subconjunto 2022-2025, alcance de la Ficha, tiene 24.779 filas.
- **Riesgos encontrados:**
  1. La fuente 2 publica el mismo proceso varias veces (filas idénticas y una por fase): se resolvió conservando la última fase de cada proceso.
  2. Unos 2.750 contratos, casi todos de contratación directa y sobre todo desde 2024, no tienen proceso publicado en el dataset: se marcan con `tiene_proceso` y se analizarán aparte.
  3. Para la limpieza: `estado_contrato` mezcla formatos (`Cerrado`, `terminado`, `cedido`) y tiene estados como `Borrador` y `Aprobado`; `modalidad_de_contratacion` tiene categorías casi duplicadas; `tipo_de_contrato` incluye `Otro` y `Decreto 092 de 2017`; 2 contratos inician después de la fecha de consulta y 3 terminan antes de iniciar.
- **Próximos pasos:** limpiar categorías y avanzar al análisis exploratorio (`02_eda`): concentración de proveedores por dependencia y comparación de `valor_del_contrato` contra `precio_base`.